<a href="https://colab.research.google.com/github/springboardmentor1/agritield-predictor/blob/Mangesh/Agriyield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Database/dataset.csv')

In [ ]:
df

,State,Crop,Soil_Type,Fertilizer,N,P,K,Rainfall_mm,Temperature_C,Yield_kg_per_acre,Soil_pH,Year
0,Karnataka,Soybean,Loamy,DAP,96,41,51,123,31.06,1899,6.82,2003
1,Odisha,Cotton,Red Soil,Urea,29,36,112,247,33.97,1002,6.41,2002
2,Punjab,Groundnut,Red Soil,Compost,37,38,177,142,24.21,1465,7.06,2015
3,Gujarat,Wheat,Red Soil,Compost,58,77,129,227,30.85,2273,5.93,2022
4,Andhra Pradesh,Cotton,Clay,Organic,108,61,63,263,37.81,1497,6.24,2017
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,Haryana,Soybean,Sandy,Compost,133,44,57,171,22.98,1091,7.63,2018
1496,Punjab,Sugarcane,Loamy,Urea,108,86,37,115,27.79,54122,5.72,2005
1497,West Bengal,Pulses,Red Soil,Urea,73,110,38,199,25.56,1784,6.66,2014
1498,Rajasthan,Maize,Loamy,NPK,101,32,117,70,33.28,1393,6.22,2007


In [ ]:
df.columns = df.columns.str.strip().str.replace(" ", "_")

In [ ]:
df.columns

Index(['State', 'Crop', 'Soil_Type', 'Fertilizer', 'N', 'P', 'K',
       'Rainfall_mm', 'Temperature_C', 'Yield_kg_per_acre', 'Soil_pH', 'Year'],
      dtype='object')

In [ ]:
print(df.head())

            State       Crop Soil_Type Fertilizer    N   P    K  Rainfall_mm  \
0       Karnataka    Soybean     Loamy        DAP   96  41   51          123   
1          Odisha     Cotton  Red Soil       Urea   29  36  112          247   
2          Punjab  Groundnut  Red Soil    Compost   37  38  177          142   
3         Gujarat      Wheat  Red Soil    Compost   58  77  129          227   
4  Andhra Pradesh     Cotton      Clay    Organic  108  61   63          263   

   Temperature_C  Yield_kg_per_acre  Soil_pH  Year  
0          31.06               1899     6.82  2003  
1          33.97               1002     6.41  2002  
2          24.21               1465     7.06  2015  
3          30.85               2273     5.93  2022  
4          37.81               1497     6.24  2017  


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   State              1500 non-null   object 
 1   Crop               1500 non-null   object 
 2   Soil_Type          1500 non-null   object 
 3   Fertilizer         1500 non-null   object 
 4   N                  1500 non-null   int64  
 5   P                  1500 non-null   int64  
 6   K                  1500 non-null   int64  
 7   Rainfall_mm        1500 non-null   int64  
 8   Temperature_C      1500 non-null   float64
 9   Yield_kg_per_acre  1500 non-null   int64  
 10  Soil_pH            1500 non-null   float64
 11  Year               1500 non-null   int64  
dtypes: float64(2), int64(6), object(4)
memory usage: 140.8+ KB
None


In [ ]:
print(df.describe())

                 N            P            K  Rainfall_mm  Temperature_C  \
count  1500.000000  1500.000000  1500.000000  1500.000000    1500.000000   
mean     73.438000    61.282667   102.112667   176.806000      28.148547   
std      37.756932    33.257422    54.362863    73.776182       5.783160   
min      10.000000     5.000000    10.000000    50.000000      18.020000   
25%      41.000000    33.000000    56.000000   113.000000      23.132500   
50%      70.000000    61.000000   100.000000   176.000000      28.490000   
75%     106.000000    91.000000   149.000000   242.000000      33.180000   
max     139.000000   119.000000   199.000000   299.000000      37.960000   

       Yield_kg_per_acre      Soil_pH         Year  
count        1500.000000  1500.000000  1500.000000  
mean         7881.892667     6.766347  2012.090000  
std         20073.333665     0.724898     7.393218  
min           502.000000     5.500000  2000.000000  
25%          1202.750000     6.120000  2005.000000

In [ ]:
print(df.isnull().sum())

State                0
Crop                 0
Soil_Type            0
Fertilizer           0
N                    0
P                    0
K                    0
Rainfall_mm          0
Temperature_C        0
Yield_kg_per_acre    0
Soil_pH              0
Year                 0
dtype: int64


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipython-input-3660/2373410678.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
#Check outliers
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = ((df[num_cols] < (Q1 - 1.5 * IQR)) |
            (df[num_cols] > (Q3 + 1.5 * IQR)))

print("Outlier Count per Column:\n", outliers.sum())

Outlier Count per Column:
 N                      0
P                      0
K                      0
Rainfall_mm            0
Temperature_C          0
Yield_kg_per_acre    131
Soil_pH                0
Year                   0
dtype: int64


In [ ]:
df = df[~((df[num_cols] < (Q1 - 1.5 * IQR)) |
          (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

In [ ]:
X = df.drop(columns=["Yield_kg_per_acre"])   # Target removed
y = df["Yield_kg_per_acre"]

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame (optional but cleaner)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [ ]:
X_train_scaled

,N,P,K,Rainfall_mm,Temperature_C,Soil_pH,Year,State_Bihar,State_Gujarat,State_Haryana,...,Crop_Wheat,Soil_Type_Clay,Soil_Type_Loamy,Soil_Type_Red Soil,Soil_Type_Sandy,Fertilizer_DAP,Fertilizer_NPK,Fertilizer_Organic,Fertilizer_Potash,Fertilizer_Urea
0,0.123114,1.348638,-1.242064,1.341548,-1.450586,1.416938,-0.818675,-0.259329,-0.251211,-0.302861,...,2.748579,-0.529813,-0.477052,-0.512812,2.182466,-0.447949,-0.427207,2.277376,-0.465475,-0.461112
1,-1.228698,-1.256088,0.792006,0.987845,1.238938,-0.496343,1.078858,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,-0.465475,-0.461112
2,-1.202192,-1.256088,0.663731,-0.250114,1.041305,-1.569983,1.349934,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,2.182466,-0.447949,-0.427207,-0.439102,-0.465475,-0.461112
3,1.501433,-0.650338,-1.498613,0.511707,-0.867999,1.100352,-1.360827,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,-0.458197,-0.447949,2.340788,-0.439102,-0.465475,-0.461112
4,-0.460021,-1.619539,1.744904,-0.399757,-0.761449,-0.455049,0.807782,3.856108,-0.251211,-0.302861,...,-0.363824,-0.529813,2.096208,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,2.148345,-0.461112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,1.448421,1.257776,-0.362466,0.212420,0.292019,-0.647754,-0.954213,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,2.182466,-0.447949,-0.427207,2.277376,-0.465475,-0.461112
1091,-0.327490,0.015987,1.231805,-1.692132,-0.397117,1.348115,1.349934,-0.259329,-0.251211,3.301844,...,-0.363824,1.887459,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,2.277376,-0.465475,-0.461112
1092,-1.493760,-1.195513,-1.406988,1.001449,-0.245885,1.155410,1.621010,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,-0.458197,2.232399,-0.427207,-0.439102,-0.465475,-0.461112
1093,-1.573278,0.954900,-0.600690,-0.195698,-1.149840,0.728707,-1.360827,-0.259329,3.980723,-0.302861,...,2.748579,1.887459,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,2.148345,-0.461112


In [ ]:
X_test_scaled

,N,P,K,Rainfall_mm,Temperature_C,Soil_pH,Year,State_Bihar,State_Gujarat,State_Haryana,...,Crop_Wheat,Soil_Type_Clay,Soil_Type_Loamy,Soil_Type_Red Soil,Soil_Type_Sandy,Fertilizer_DAP,Fertilizer_NPK,Fertilizer_Organic,Fertilizer_Potash,Fertilizer_Urea
0,-0.804600,0.288575,-0.179216,-0.807875,0.266241,-1.074457,0.401168,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,-0.465475,-0.461112
1,1.183359,-1.286376,1.250130,-0.154886,-1.493549,-0.647754,0.807782,3.856108,-0.251211,-0.302861,...,2.748579,1.887459,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,-0.465475,-0.461112
2,1.077335,-0.983501,-0.124242,-0.141282,-0.988297,-1.528689,-1.089751,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,2.096208,-0.512812,-0.458197,2.232399,-0.427207,-0.439102,-0.465475,-0.461112
3,-0.513033,-0.892638,-1.425313,1.028657,0.587609,-0.689048,-1.631903,3.856108,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,1.950034,-0.458197,-0.447949,2.340788,-0.439102,-0.465475,-0.461112
4,-0.009416,0.591450,-0.802265,-1.610508,-0.505386,-0.207286,1.621010,3.856108,-0.251211,-0.302861,...,-0.363824,-0.529813,2.096208,-0.512812,-0.458197,2.232399,-0.427207,-0.439102,-0.465475,-0.461112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,0.361669,-0.347463,0.223933,-0.658232,-0.178862,1.086587,-0.818675,-0.259329,-0.251211,-0.302861,...,-0.363824,1.887459,-0.477052,-0.512812,-0.458197,-0.447949,2.340788,-0.439102,-0.465475,-0.461112
270,-0.937131,-1.619539,1.744904,0.742974,0.525741,0.687413,0.130092,3.856108,-0.251211,-0.302861,...,-0.363824,1.887459,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,-0.465475,2.168669
271,-0.725082,0.106850,1.561654,-0.522193,0.994904,0.563532,1.078858,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,-0.458197,-0.447949,-0.427207,-0.439102,-0.465475,2.168669
272,1.501433,0.167425,0.535457,1.463983,0.527460,1.265527,-1.631903,-0.259329,-0.251211,-0.302861,...,-0.363824,-0.529813,-0.477052,-0.512812,2.182466,-0.447949,-0.427207,-0.439102,-0.465475,2.168669


In [ ]:
models = {
    "Random Forest": RandomForestRegressor(n_estimators=500, random_state=42),
    "Linear Regression": LinearRegression(),
    "XGBoost": XGBRegressor(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
}

In [ ]:
results = {}

for name, model in models.items():

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}

    print(f"\n{name}")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2 Score:", r2)


Random Forest
MAE: 365.4018321167884
RMSE: 460.1621555589474
R2 Score: 0.6559019261388596

Linear Regression
MAE: 365.38911895904903
RMSE: 446.78196728908256
R2 Score: 0.675621757619306

XGBoost
MAE: 403.63494873046875
RMSE: 514.0070828792149
R2 Score: 0.5706627368927002


In [ ]:
results_df = pd.DataFrame(results).T
print("\nModel Comparison:\n")
print(results_df.sort_values(by="R2", ascending=False))


Model Comparison:

                          MAE        RMSE        R2
Linear Regression  365.389119  446.781967  0.675622
Random Forest      365.401832  460.162156  0.655902
XGBoost            403.634949  514.007083  0.570663
